#  Student Health Monitoring and Early Detection Using Machine Learning

**Training notebook** — Phase 1 (Data Analysis & Evaluation) and Phase 2 (Final Model & Artifacts).

The final deployed model is a **Support Vector Machine (SVM)**, which outperformed a Random Forest baseline.

This notebook produces the artifacts used by the Streamlit app:
`model.pkl`, `scaler.pkl`, `encoders.pkl`, and `feature_metadata.json`.

## 1. Imports & Configuration

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_score,
                             recall_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC

DATA_PATH = 'Student_Depression_Dataset.csv'
TARGET = 'Depression'
RANDOM_STATE = 42
TEST_SIZE = 0.2

# Non-predictive / dirty columns removed before modelling (see README).
DROP_COLS = ['id', 'City', 'Profession', 'Work Pressure', 'Job Satisfaction']

## 2. Load the dataset & display information

In [ ]:
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

## 3. Detect missing values

In [ ]:
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else 'No missing values.')
print('\nTarget balance:')
print(df[TARGET].value_counts())

## 4. Clean data: drop columns & handle missing values

- Numeric columns → **median** imputation.
- Categorical columns → **mode** imputation.

In [ ]:
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

for col in df.select_dtypes(include='number').columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include='object').columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode().iloc[0])

print('Remaining missing values:', int(df.isnull().sum().sum()))
df.head()

## 5. Identify categorical columns & apply Label Encoding

In [ ]:
categorical_cols = [c for c in df.select_dtypes(include='object').columns if c != TARGET]
print('Categorical columns:', categorical_cols)

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le
df.head()

## 6. Separate X / y, split, and scale

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
feature_order = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('Train:', X_train_scaled.shape, ' Test:', X_test_scaled.shape)

## 7. Train SVM (and a Random Forest baseline for comparison)

In [ ]:
svm = SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE)
svm.fit(X_train_scaled, y_train)

rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train_scaled, y_train)
print('Models trained.')

## 8. Evaluate: Accuracy, Precision, Recall, F1, Confusion Matrix

In [ ]:
def evaluate(name, y_true, y_pred):
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
    }
    print(f'--- {name} ---')
    for k, v in metrics.items():
        print(f'{k.capitalize():<10}: {v:.4f}')
    print('Confusion Matrix:')
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, digits=4))
    return metrics

svm_metrics = evaluate('SVM', y_test, svm.predict(X_test_scaled))
rf_metrics = evaluate('Random Forest', y_test, rf.predict(X_test_scaled))

In [ ]:
comparison = pd.DataFrame({'SVM': svm_metrics, 'Random Forest': rf_metrics}).T.round(4)
print(comparison)
print('\nBest by F1:', 'SVM' if svm_metrics['f1'] >= rf_metrics['f1'] else 'Random Forest')

## 9. Phase 2 — Final model on the full dataset + save artifacts

In [ ]:
# Scale on the full cleaned dataset and train the final SVM.
final_scaler = StandardScaler()
X_full_scaled = final_scaler.fit_transform(X[feature_order])

final_svm = SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE)
final_svm.fit(X_full_scaled, y)
print('Final SVM trained on the complete cleaned dataset.')

In [ ]:
# Build feature metadata so the Streamlit app can auto-generate its inputs.
raw_df = pd.read_csv(DATA_PATH)
metadata_features = []
for col in feature_order:
    if col in encoders:
        options = list(encoders[col].classes_)
        metadata_features.append({'name': col, 'type': 'categorical',
                                  'options': options, 'default': options[0]})
    else:
        s = raw_df[col].dropna()
        is_int = bool(np.all(np.equal(np.mod(s.values, 1), 0)))
        metadata_features.append({'name': col, 'type': 'numeric',
                                  'min': float(s.min()), 'max': float(s.max()),
                                  'default': float(s.median()),
                                  'is_integer': is_int,
                                  'step': 1.0 if is_int else 0.01})

joblib.dump(final_svm, 'model.pkl')
joblib.dump(final_scaler, 'scaler.pkl')
joblib.dump(encoders, 'encoders.pkl')
with open('feature_metadata.json', 'w', encoding='utf-8') as fh:
    json.dump({'target': TARGET, 'feature_order': feature_order,
               'features': metadata_features}, fh, indent=2)

print('Saved: model.pkl, scaler.pkl, encoders.pkl, feature_metadata.json')

✅ **Done.** The Streamlit app (`app.py`) can now be launched with `streamlit run app.py`.